## load dependacies

In [1]:
import pandas as pd
import numpy as np

In [3]:
users = pd.read_csv('../../data/preproccedData/Augmented_PreProccedNutrationParameters.csv')
food_df = pd.read_csv('../../data/RecommandationDatasets/NutritionsDatasets/SL_Diabetic_Foods.csv')  


In [4]:
users.head()

,Age,Gender,Height,Weight,Carbohydrate_Consumption,Protein_Intake,Fat_Intake,Regularity_of_Meals,Portion_Control,Hydration,Caloric_Balance,Sugar_Consumption,BMI,DiabetesRisk,NutritionRisk
0,24,1,161.609263,48.731866,1.852525,1.090620,0.916151,0.959783,1.083803,2.043815,2120.952023,1.061643,17.093053,30.774236,66.860735
1,24,1,159.653670,53.637920,0.933021,2.175717,0.904236,0.008224,1.065243,1.038885,2120.958294,1.052971,19.227524,32.703477,33.419361
2,28,0,148.049594,46.770525,2.773378,3.266809,0.920246,0.000000,2.150421,2.047019,2120.920320,1.051036,19.532064,28.756909,16.708769
3,24,0,157.719943,54.580482,1.856103,1.071392,0.915962,0.960743,3.224980,5.128280,1387.532090,2.099957,20.091964,37.049640,33.412868
4,22,0,149.030373,44.849854,1.855097,1.116779,0.916990,0.950166,2.145013,3.079390,2120.942655,3.180002,18.476682,24.567752,16.712636


## user df explanation

| Field                         | Input Type | Possible Values                        | Real-world Interpretation                                |
| ----------------------------- | ---------- | -------------------------------------- | -------------------------------------------------------- |
| **Carbohydrate\_Consumption** | Select     | 1.0 to 5.0 (servings/day)              | Daily intake of carbs like rice, pasta, bread            |
| **Protein\_Intake**           | Select     | "Yes", "No"                            | Whether protein-rich foods are consumed regularly        |
| **Fat\_Intake**               | Select     | "Healthy fats", "Unhealthy fats"       | Quality of fat in diet (e.g., olive oil vs. fried foods) |
| **Regularity\_of\_Meals**     | Select     | "Yes", "No"                            | Whether the person skips meals                           |
| **Portion\_Control**          | Select     | 1.0 to 5.0 (mapped to portion sizes)   | Size of each meal in cups/plates                         |
| **Caloric\_Balance**          | Input      | Numeric (e.g., 2000)                   | Daily calorie intake (if known)                          |
| **Sugar\_Consumption**        | Select     | 1.0 to 5.0 (g/day or portions per day) | Frequency of sugary foods or drinks                      |




| Field | Input Type | Possible Values | Real-world Interpretation |
|-------|------------|-----------------|---------------------------|
| **Age** | Input | Numeric (e.g., 24) | User's age in years |
| **Gender** | Select | 0 (Female), 1 (Male) | User's gender |
| **Height** | Input | Numeric (cm) | User's height in centimeters |
| **Weight** | Input | Numeric (kg) | User's weight in kilograms |
| **Carbohydrate_Consumption** | Select | 1.0 to 5.0 (servings/day) | Daily intake of carbs like rice, pasta, bread |
| **Protein_Intake** | Select | "Yes", "No"| Whether protein-rich foods are consumed |
| **Fat_Intake** | Select | "Healthy fats", "Unhealthy fats" 1 or 0(servings/day) | Quality of fat in diet (e.g., olive oil vs. fried) |
| **Regularity_of_Meals** | Select | "Yes", "No" "1.0" or 0.0 | How regularly meals are consumed (0 = irregular, 5 = very regular) |
| **Portion_Control** | Select | 1.0 to 5.0 | Size of each meal in cups/plates |
| **Caloric_Balance** | Input | Numeric (e.g., 2120) | Daily calorie intake |
| **Sugar_Consumption** | Select | 1.0 to 5.0 | Frequency of sugary foods or drinks |
| **BMI** | Calculated | Numeric | Body Mass Index (calculated from height and weight) |
| **DiabetesRisk** | Predicted | Numeric (0-100) | Risk score for diabetes (higher = higher risk) |
| **NutritionRisk** | Predicted | Numeric (0-100) | Risk score for nutrition-related issues (higher = higher risk) |



In [5]:
food_df.head()

,Food Name,Ingredients,SuitableForDiabetic,Calories_(kcal),Carbohydrate_(g),Protein_(g),Fat_(g),Fiber_(g),Free_sugars_(g),Cholesterol_(mg),Sodium_(mg)
0,ALMONDS,['ALMONDS'],True,180.0,6.0,6.0,15.0,13.1,2.196241,0.0,29.137727
1,AMBUL KESEL,['AMBUL KESEL'],True,45.0,11.0,0.5,0.1,NaN,0.973530,0.0,35.511988
2,AMBUN KESEL,['AMBUN KESEL'],True,90.0,23.0,1.3,0.3,NaN,0.973530,0.0,35.511988
3,APPLE,['APPLE'],True,80.0,22.0,0.0,0.0,NaN,0.973530,0.0,35.511988
4,AVOCADO,['AVOCADO'],True,80.0,4.3,1.0,7.4,NaN,0.973530,0.0,35.511988


In [8]:
# ... existing code ...

# Calculate quantity and GI index based on nutrients
def calculate_quantity_and_gi(row):
    # Calculate quantity based on total nutrients
    total_nutrients = row['Calories_(kcal)'] + row['Protein_(g)'] + row['Carbohydrate_(g)'] + row['Fat_(g)']
    quantity = total_nutrients / 100  # Normalize to 100g serving
    
    # Calculate GI index based on carbohydrate content and type
    # This is a simplified calculation - actual GI depends on many factors
    base_gi = 50  # Base GI value
    carb_factor = row['Carbohydrate_(g)'] / 100  # Normalize carbs
    fiber_factor = 1 - (row['Fiber_(g)'] / 100) if 'Fiber_(g)' in row else 0.5  # Fiber reduces GI
    
    gi_index = base_gi * carb_factor * fiber_factor
    
    return pd.Series([quantity, gi_index])

# Apply the calculation to the dataframe
food_df[['Quantity', 'GI_Index']] = food_df.apply(calculate_quantity_and_gi, axis=1)

# Display the first few rows with new columns
print("\nFood DataFrame with Quantity and GI Index:")
print(food_df[['Food Name', 'Quantity', 'GI_Index']].head())

# ... existing code ...


Food DataFrame with Quantity and GI Index:
     Food Name  Quantity  GI_Index
0      ALMONDS     2.070     2.607
1  AMBUL KESEL     0.566       NaN
2  AMBUN KESEL     1.146       NaN
3        APPLE     1.020       NaN
4      AVOCADO     0.927       NaN


In [25]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import torch
import pickle
from transformers import BertTokenizer, BertModel

# Step 2: Load the data
users = pd.read_csv('../../data/preproccedData/Augmented_PreProccedNutrationParameters.csv')
food_df = pd.read_csv('../../data/RecommandationDatasets/NutritionsDatasets/pred_food.csv')

# Display the first few rows of each dataset
print("Users DataFrame:")
display(users.head())
print("\nFood DataFrame:")
display(food_df.head())

Users DataFrame:


,Age,Gender,Height,Weight,Carbohydrate_Consumption,Protein_Intake,Fat_Intake,Regularity_of_Meals,Portion_Control,Hydration,Caloric_Balance,Sugar_Consumption,BMI,DiabetesRisk,NutritionRisk
0,24,1,161.609263,48.731866,1.852525,1.090620,0.916151,0.959783,1.083803,2.043815,2120.952023,1.061643,17.093053,30.774236,66.860735
1,24,1,159.653670,53.637920,0.933021,2.175717,0.904236,0.008224,1.065243,1.038885,2120.958294,1.052971,19.227524,32.703477,33.419361
2,28,0,148.049594,46.770525,2.773378,3.266809,0.920246,0.000000,2.150421,2.047019,2120.920320,1.051036,19.532064,28.756909,16.708769
3,24,0,157.719943,54.580482,1.856103,1.071392,0.915962,0.960743,3.224980,5.128280,1387.532090,2.099957,20.091964,37.049640,33.412868
4,22,0,149.030373,44.849854,1.855097,1.116779,0.916990,0.950166,2.145013,3.079390,2120.942655,3.180002,18.476682,24.567752,16.712636



Food DataFrame:


,Food Name,Glycemic Index,Calories,Carbohydrates,Protein,Fat,Suitable for Diabetes,Suitable for Blood Pressure,Sodium Content,Potassium Content,Magnesium Content,Calcium Content,Fiber Content
0,Apple,39,52,14.0,0.3,0.2,1,1,0,107,9,6,2.4
1,Banana,51,96,23.0,1.1,0.2,1,1,1,358,27,5,2.6
2,Orange,42,43,9.0,0.9,0.1,1,1,0,181,10,40,2.3
3,Strawberries,40,29,7.0,0.7,0.3,1,1,1,153,13,16,2.0
4,Blueberries,53,57,14.0,0.7,0.3,1,1,1,77,9,6,2.4


In [27]:
# Step 3: Define the EnhancedFoodPreferenceMatcher class
class EnhancedFoodPreferenceMatcher:
    def __init__(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.model = BertModel.from_pretrained('bert-base-uncased')
        
    def encode_food_description(self, food_name, nutritional_info):
        description = f"{food_name} has {nutritional_info['Carbohydrates']}g carbs, " \
                     f"glycemic index of {nutritional_info['Glycemic Index']}, " \
                     f"{nutritional_info['Protein']}g protein, " \
                     f"{nutritional_info['Fat']}g fat, " \
                     f"{nutritional_info['Fiber Content']}g fiber, " \
                     f"{nutritional_info['Calories']} calories, " \
                     f"and {nutritional_info['Potassium Content']}mg potassium"
        return self.tokenizer(description, return_tensors='pt', padding=True, truncation=True)
    
    def encode_user_scenario(self, user_profile):
        scenarios = self._analyze_user_scenarios(user_profile)
        scenario_description = self._create_scenario_description(scenarios)
        return self.tokenizer(scenario_description, return_tensors='pt', padding=True, truncation=True)
    
    def _analyze_user_scenarios(self, user_profile):
        scenarios = {
            'diabetes': {
                'risk_level': 'high' if user_profile['DiabetesRisk'] > 50 else 'moderate' if user_profile['DiabetesRisk'] > 30 else 'low',
                'sugar_level': user_profile['Sugar_Consumption'],
                'carb_level': user_profile['Carbohydrate_Consumption']
            },
            'nutrition': {
                'risk_level': 'high' if user_profile['NutritionRisk'] > 50 else 'moderate' if user_profile['NutritionRisk'] > 30 else 'low',
                'protein_level': user_profile['Protein_Intake'],
                'fat_level': user_profile['Fat_Intake']
            },
            'meal_pattern': {
                'regularity': user_profile['Regularity_of_Meals'],
                'portion': user_profile['Portion_Control'],
                'hydration': user_profile['Hydration']
            }
        }
        return scenarios
    
    def _create_scenario_description(self, scenarios):
        description_parts = []
        
        if scenarios['diabetes']['risk_level'] in ['high', 'moderate']:
            description_parts.append(
                f"User has {scenarios['diabetes']['risk_level']} diabetes risk, "
                f"consumes {scenarios['diabetes']['sugar_level']} sugar and "
                f"{scenarios['diabetes']['carb_level']} carbs"
            )
        
        if scenarios['nutrition']['risk_level'] in ['high', 'moderate']:
            description_parts.append(
                f"User has {scenarios['nutrition']['risk_level']} nutrition risk, "
                f"consumes {scenarios['nutrition']['protein_level']} protein and "
                f"{scenarios['nutrition']['fat_level']} fat"
            )
        
        description_parts.append(
            f"User has {scenarios['meal_pattern']['regularity']} meal regularity, "
            f"{scenarios['meal_pattern']['portion']} portion control, and "
            f"{scenarios['meal_pattern']['hydration']} hydration level"
        )
        
        return " ".join(description_parts)
    
    def calculate_similarity(self, food_encoding, user_encoding):
        with torch.no_grad():
            food_output = self.model(**food_encoding)
            user_output = self.model(**user_encoding)
        
        food_embedding = food_output.last_hidden_state[:, 0, :]
        user_embedding = user_output.last_hidden_state[:, 0, :]
        
        similarity = torch.nn.functional.cosine_similarity(food_embedding, user_embedding)
        return similarity.item()

In [29]:
# Step 4: Define the MealPlanRecommender class
class MealPlanRecommender:
    def __init__(self, food_df, user_profile):
        self.food_df = food_df
        self.user_profile = user_profile
        self.matcher = EnhancedFoodPreferenceMatcher()
        
    def get_meal_plan(self, num_days=7):
        meal_plans = []
        
        for day in range(num_days):
            daily_plan = {
                'breakfast': self._get_meal_recommendations('breakfast'),
                'lunch': self._get_meal_recommendations('lunch'),
                'dinner': self._get_meal_recommendations('dinner'),
                'snacks': self._get_meal_recommendations('snack')
            }
            meal_plans.append(daily_plan)
            
        return meal_plans
    
    def _get_meal_recommendations(self, meal_type, top_n=2):
        recommendations = []
        
        for _, food in self.food_df.iterrows():
            if self._is_suitable_for_meal(food, meal_type):
                food_encoding = self.matcher.encode_food_description(
                    food['Food Name'], 
                    food.to_dict()
                )
                user_encoding = self.matcher.encode_user_scenario(self.user_profile)
                similarity = self.matcher.calculate_similarity(food_encoding, user_encoding)
                
                if self._passes_meal_filters(food, meal_type):
                    recommendations.append({
                        'food_name': food['Food Name'],
                        'similarity_score': similarity,
                        'nutritional_info': food.to_dict(),
                        'meal_type': meal_type
                    })
        
        recommendations.sort(key=lambda x: x['similarity_score'], reverse=True)
        return recommendations[:top_n]
    
    def _is_suitable_for_meal(self, food, meal_type):
        meal_criteria = {
            'breakfast': {
                'min_calories': 200,
                'max_calories': 400,
                'min_protein': 5,
                'min_fiber': 2
            },
            'lunch': {
                'min_calories': 400,
                'max_calories': 600,
                'min_protein': 10,
                'min_fiber': 3
            },
            'dinner': {
                'min_calories': 400,
                'max_calories': 600,
                'min_protein': 15,
                'min_fiber': 3
            },
            'snack': {
                'min_calories': 100,
                'max_calories': 200,
                'min_protein': 2,
                'min_fiber': 1
            }
        }
        
        criteria = meal_criteria[meal_type]
        return (criteria['min_calories'] <= food['Calories'] <= criteria['max_calories'] and
                food['Protein'] >= criteria['min_protein'] and
                food['Fiber Content'] >= criteria['min_fiber'])
    
    def _passes_meal_filters(self, food, meal_type):
        scenarios = self.matcher._analyze_user_scenarios(self.user_profile)
        
        if scenarios['diabetes']['risk_level'] in ['high', 'moderate']:
            if not (food['Glycemic Index'] < 55 and food['Suitable for Diabetes'] == 1):
                return False
        
        if scenarios['nutrition']['risk_level'] in ['high', 'moderate']:
            if not (food['Protein'] > 5 and food['Fiber Content'] > 2):
                return False
        
        return True

In [30]:
# Step 5: Define utility functions for saving and loading
def save_recommendation_system(food_df, user_profiles, save_path='recommendation_system'):
    system_components = {
        'food_df': food_df,
        'user_profiles': user_profiles,
        'matcher': EnhancedFoodPreferenceMatcher(),
        'meal_planner': MealPlanRecommender(food_df, user_profiles.iloc[0].to_dict())
    }
    
    with open(f'{save_path}_components.pkl', 'wb') as f:
        pickle.dump(system_components, f)
    
    print(f"Recommendation system components saved to {save_path}_components.pkl")

def load_recommendation_system(load_path='recommendation_system'):
    try:
        with open(f'{load_path}_components.pkl', 'rb') as f:
            system_components = pickle.load(f)
        print("Successfully loaded recommendation system components")
        return system_components
    except FileNotFoundError:
        print(f"Error: Could not find {load_path}_components.pkl")
        return None

In [31]:
# Step 6: Define functions for printing meal plans
def print_meal_plan(meal_plans):
    for day, plan in enumerate(meal_plans, 1):
        print(f"\n=== Day {day} ===")
        
        for meal_type, foods in plan.items():
            print(f"\n{meal_type.title()}:")
            for food in foods:
                print(f"\nFood: {food['food_name']}")
                print(f"Calories: {food['nutritional_info']['Calories']}")
                print(f"Protein: {food['nutritional_info']['Protein']}g")
                print(f"Carbs: {food['nutritional_info']['Carbohydrates']}g")
                print(f"Fat: {food['nutritional_info']['Fat']}g")
                print(f"Fiber: {food['nutritional_info']['Fiber Content']}g")
                print(f"Glycemic Index: {food['nutritional_info']['Glycemic Index']}")

def get_personalized_meal_plan(user_profile, food_df):
    meal_planner = MealPlanRecommender(food_df, user_profile)
    meal_plans = meal_planner.get_meal_plan(num_days=7)
    
    print("User Health Profile:")
    scenarios = meal_planner.matcher._analyze_user_scenarios(user_profile)
    for scenario, details in scenarios.items():
        print(f"\n{scenario.title()}:")
        for key, value in details.items():
            print(f"{key}: {value}")
    
    print("\n=== Personalized 7-Day Meal Plan ===")
    print_meal_plan(meal_plans)

In [32]:
# Step 7: Save the system
save_recommendation_system(food_df, users)

Recommendation system components saved to recommendation_system_components.pkl


In [22]:
# Step 8: Load and use the system
loaded_components = load_recommendation_system()

if loaded_components:
    # Get meal plan for first user
    user_profile = loaded_components['user_profiles'].iloc[0].to_dict()
    get_personalized_meal_plan(user_profile, loaded_components['food_df'])

Successfully loaded recommendation system components
User Health Profile:

Diabetes:
risk_level: moderate
sugar_level: 1.0616433422093456
carb_level: 1.8525247515444565

Nutrition:
risk_level: high
protein_level: 1.090619841804255
fat_level: 0.9161511386735988

Meal_Pattern:
regularity: 0.9597825377554642
portion: 1.0838033309574044
hydration: 2.04381457918106

=== Personalized 7-Day Meal Plan ===

=== Day 1 ===

Breakfast:

Lunch:

Dinner:

Snacks:

=== Day 2 ===

Breakfast:

Lunch:

Dinner:

Snacks:

=== Day 3 ===

Breakfast:

Lunch:

Dinner:

Snacks:

=== Day 4 ===

Breakfast:

Lunch:

Dinner:

Snacks:

=== Day 5 ===

Breakfast:

Lunch:

Dinner:

Snacks:

=== Day 6 ===

Breakfast:

Lunch:

Dinner:

Snacks:

=== Day 7 ===

Breakfast:

Lunch:

Dinner:

Snacks:


In [33]:
class MealPlanRecommender:
    def __init__(self, food_df, user_profile):
        self.food_df = food_df
        self.user_profile = user_profile
        self.matcher = EnhancedFoodPreferenceMatcher()
        
    def get_meal_plan(self, num_days=7):
        """Generate a meal plan for the specified number of days"""
        meal_plans = []
        
        for day in range(num_days):
            daily_plan = {
                'breakfast': self._get_meal_recommendations('breakfast'),
                'lunch': self._get_meal_recommendations('lunch'),
                'dinner': self._get_meal_recommendations('dinner'),
                'snacks': self._get_meal_recommendations('snack')
            }
            meal_plans.append(daily_plan)
            
        return meal_plans
        
    def _is_suitable_for_meal(self, food, meal_type):
        # More lenient meal criteria
        meal_criteria = {
            'breakfast': {
                'min_calories': 50,  # Lowered from 200
                'max_calories': 500,  # Increased from 400
                'min_protein': 0.5,   # Lowered from 5
                'min_fiber': 0.5      # Lowered from 2
            },
            'lunch': {
                'min_calories': 100,  # Lowered from 400
                'max_calories': 800,  # Increased from 600
                'min_protein': 1,     # Lowered from 10
                'min_fiber': 1        # Lowered from 3
            },
            'dinner': {
                'min_calories': 100,  # Lowered from 400
                'max_calories': 800,  # Increased from 600
                'min_protein': 1,     # Lowered from 15
                'min_fiber': 1        # Lowered from 3
            },
            'snack': {
                'min_calories': 20,   # Lowered from 100
                'max_calories': 300,  # Increased from 200
                'min_protein': 0.3,   # Lowered from 2
                'min_fiber': 0.3      # Lowered from 1
            }
        }
        
        criteria = meal_criteria[meal_type]
        return (criteria['min_calories'] <= food['Calories'] <= criteria['max_calories'] and
                food['Protein'] >= criteria['min_protein'] and
                food['Fiber Content'] >= criteria['min_fiber'])
    
    def _passes_meal_filters(self, food, meal_type):
        scenarios = self.matcher._analyze_user_scenarios(self.user_profile)
        
        # More lenient diabetes filtering
        if scenarios['diabetes']['risk_level'] in ['high', 'moderate']:
            if not (food['Glycemic Index'] < 70):  # Increased from 55
                return False
        
        # More lenient nutrition filtering
        if scenarios['nutrition']['risk_level'] in ['high', 'moderate']:
            if not (food['Protein'] > 0.5 and food['Fiber Content'] > 0.5):  # Lowered from 5 and 2
                return False
        
        return True

    def _get_meal_recommendations(self, meal_type, top_n=2):
        recommendations = []
        
        for _, food in self.food_df.iterrows():
            if self._is_suitable_for_meal(food, meal_type):
                food_encoding = self.matcher.encode_food_description(
                    food['Food Name'], 
                    food.to_dict()
                )
                user_encoding = self.matcher.encode_user_scenario(self.user_profile)
                similarity = self.matcher.calculate_similarity(food_encoding, user_encoding)
                
                if self._passes_meal_filters(food, meal_type):
                    recommendations.append({
                        'food_name': food['Food Name'],
                        'similarity_score': similarity,
                        'nutritional_info': food.to_dict(),
                        'meal_type': meal_type
                    })
        
        recommendations.sort(key=lambda x: x['similarity_score'], reverse=True)
        return recommendations[:top_n]

# Test the modified system
def test_meal_plan():
    # Load the food data
    food_df = pd.read_csv('../../data/RecommandationDatasets/NutritionsDatasets/pred_food.csv')
    
    # Create a test user profile
    test_user_profile = {
        'Age': 35,
        'Gender': 1,
        'Height': 175.0,
        'Weight': 75.0,
        'Carbohydrate_Consumption': 2.5,
        'Protein_Intake': 1.8,
        'Fat_Intake': 1.2,
        'Regularity_of_Meals': 0.8,
        'Portion_Control': 1.5,
        'Hydration': 2.5,
        'Caloric_Balance': 2200.0,
        'Sugar_Consumption': 1.5,
        'BMI': 24.5,
        'DiabetesRisk': 35.0,
        'NutritionRisk': 45.0
    }
    
    # Create meal planner with test profile
    meal_planner = MealPlanRecommender(food_df, test_user_profile)
    meal_plans = meal_planner.get_meal_plan(num_days=7)
    
    # Print results
    print("Test User Health Profile:")
    scenarios = meal_planner.matcher._analyze_user_scenarios(test_user_profile)
    for scenario, details in scenarios.items():
        print(f"\n{scenario.title()}:")
        for key, value in details.items():
            print(f"{key}: {value}")
    
    print("\n=== Personalized 7-Day Meal Plan ===")
    print_meal_plan(meal_plans)

# Run the test
test_meal_plan()

KeyboardInterrupt: 